In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.data set_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [22]:
import pandas as pd
import numpy as np

# Load the primary dataset using the nested path
path = '/kaggle/input/competitions/home-credit-default-risk/application_train.csv'
df = pd.read_csv(path)

# Verify the load
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 columns:")
print(df.columns[:5].tolist())

Dataset Shape: (307511, 122)

First 5 columns:
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR']


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# 1. Select core features for the PD model
core_features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_EMPLOYED', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
X = df[core_features].copy()
y = df['TARGET']

# 2. Handle data anomaly: DAYS_EMPLOYED has a weird anomaly where positive 365243 means "unemployed"
X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, 0)

# 3. Train-Test Split (80% train, 20% validation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Impute missing values with the median of each column
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 5. Scale features (essential for Logistic Regression performance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Data cleaning and splitting complete!")
print(f"Training set size: {X_train_scaled.shape[0]} applications")
print(f"Testing set size: {X_test_scaled.shape[0]} applications")

Data cleaning and splitting complete!
Training set size: 246008 applications
Testing set size: 61503 applications


In [24]:
# 6. Initialize and train the Logistic Regression model
pd_model = LogisticRegression(max_iter=1000, random_state=42)
pd_model.fit(X_train_scaled, y_train)

# 7. Predict probabilities for the test set
# [:, 1] gives us the probability of defaulting (class 1)
y_pred_proba = pd_model.predict_proba(X_test_scaled)[:, 1]

# 8. Evaluate using ROC-AUC score
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"Model ROC-AUC Score: {auc_score:.4f}")

Model ROC-AUC Score: 0.7108


In [25]:
# 1. Define the scorecard scaling parameters
# We want a base score of 600 where the odds of paying back double every 20 points (PDO = 20)
target_score = 600
target_odds = 50  # 50:1 odds of good vs bad at 600 points
pdo = 20

factor = pdo / np.log(2)
offset = target_score - factor * np.log(target_odds)

print(f"Scorecard Scale Initialized:")
print(f"Factor: {factor:.4f} | Offset: {offset:.4f}\n")

# 2. Extract the baseline intercept score
intercept = pd_model.intercept_[0]
base_points = offset + (factor * intercept)
print(f"Base Points (Model Intercept): {base_points:.2f}\n")

# 3. Map feature coefficients to scorecard points
# Note: A negative coefficient in credit risk means increasing the variable *reduces* default risk
# (e.g., higher external credit scores reduce default probability, giving positive points)
coefficients = pd_model.coef_[0]

scorecard_features = {}
print("--- FEATURE POINTS ASSIGNMENT (Per Standard Deviation Change) ---")
for feat, coef in zip(core_features, coefficients):
    # We multiply by -1 because standard logistic regression outputs probability of BAD (default),
    # but a credit score measures the probability of GOOD (repayment).
    points = -coef * factor
    scorecard_features[feat] = points
    print(f"{feat:<20} : {points:+.2f} points")

Scorecard Scale Initialized:
Factor: 28.8539 | Offset: 487.1229

Base Points (Model Intercept): 410.08

--- FEATURE POINTS ASSIGNMENT (Per Standard Deviation Change) ---
AMT_INCOME_TOTAL     : +0.02 points
AMT_CREDIT           : +3.67 points
AMT_ANNUITY          : -3.93 points
DAYS_EMPLOYED        : -3.95 points
EXT_SOURCE_2         : +13.74 points
EXT_SOURCE_3         : +14.10 points


In [26]:
import pulp

# 1. Take a representative sample of 5,000 applicants to optimize efficiently
opt_df = X_test.copy()
opt_df['PD'] = y_pred_proba  # Our model's predicted default probabilities
opt_df = opt_df.sample(n=5000, random_state=42)

# 2. Define financial parameters
INTEREST_RATE = 0.10  # 10% return on successful loans
TOTAL_BUDGET = 50_000_000  # $50 Million total capital available to lend
MAX_PORTFOLIO_DEFAULT_RATE = 0.05  # Cap portfolio default rate at 5%

# Calculate expected financial return per applicant
opt_df['Expected_Return'] = (opt_df['AMT_CREDIT'] * INTEREST_RATE * (1 - opt_df['PD'])) - (opt_df['AMT_CREDIT'] * opt_df['PD'])

# 3. Initialize the Maximization Problem
prob = pulp.LpProblem("Credit_Portfolio_Optimization", pulp.LpMaximize)

# 4. Create Binary Decision Variables (1 = Approve, 0 = Reject)
applicant_ids = opt_df.index.tolist()
x = pulp.LpVariable.dicts("approve", applicant_ids, cat='Binary')

# 5. Objective Function: Maximize total expected return
prob += pulp.lpSum([x[i] * opt_df.loc[i, 'Expected_Return'] for i in applicant_ids])

# 6. Constraint 1: Total loan amount cannot exceed total budget
prob += pulp.lpSum([x[i] * opt_df.loc[i, 'AMT_CREDIT'] for i in applicant_ids]) <= TOTAL_BUDGET

# 7. Constraint 2: Portfolio default rate constraint
# Formulated linearly: Sum(x_i * (PD_i - Target_PD)) <= 0
prob += pulp.lpSum([x[i] * (opt_df.loc[i, 'PD'] - MAX_PORTFOLIO_DEFAULT_RATE) for i in applicant_ids]) <= 0

# 8. Solve the optimization problem
print("Running the optimization solver...")
status = prob.solve()
print(f"Solver Status: {pulp.LpStatus[status]}")

# 9. Extract and display results
approved_indices = [i for i in applicant_ids if x[i].varValue == 1]
portfolio_df = opt_df.loc[approved_indices]

total_funded = portfolio_df['AMT_CREDIT'].sum()
total_return = portfolio_df['Expected_Return'].sum()
avg_pd = portfolio_df['PD'].mean()

print("\n--- OPTIMIZATION RESULTS ---")
print(f"Applicants Approved : {len(approved_indices)} out of 5,000")
print(f"Total Capital Deployed: ${total_funded:,.2f} (Budget: ${TOTAL_BUDGET:,.2f})")
print(f"Expected Net Return  : ${total_return:,.2f}")
print(f"Portfolio Default Rate: {avg_pd * 100:.2f}% (Constraint: < {MAX_PORTFOLIO_DEFAULT_RATE * 100:.1f}%)")

Running the optimization solver...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/c05c423930864110bc32953ae0f75d1a-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/c05c423930864110bc32953ae0f75d1a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 25008 RHS
At line 25011 BOUNDS
At line 30012 ENDATA
Problem MODEL has 2 rows, 5000 columns and 10000 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 4.22725e+06 - 0.01 seconds
Cgl0004I processed model has 2 rows, 3504 columns (3504 integer (3504 of which binary)) and 7008 elements
Cbc0038I Initial state - 1 integers unsatisfied sum - 0.28955
Cbc0038I Solution found of -4.20814e+06
Cbc0038I Before mini branch and bound, 3503 integers at bound fixed and 0 continuous
Cbc